# Notebook 3 — Network Analysis and Visualisation

This notebook builds the complex network from the pre-computed Monte Carlo adjacency matrix, extracts structural and local metrics, detects communities, and produces interactive Folium maps.

> **Prerequisites:** Run Notebooks 1 and 2 first (or place the pre-computed files at the paths defined in `config.py`). The Monte Carlo simulation file `monteCarlosimulations.npy` must be available — it was produced on an HPC cluster and is not regenerated here.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pickle

from vehicle_theft_network.config import DataConfig, EventSyncConfig
from vehicle_theft_network.data.loader import load_npy, load_pickle
from vehicle_theft_network.event_sync.analyzer import build_adjacency_matrix
from vehicle_theft_network.network.builder import build_network, get_coord_labels
from vehicle_theft_network.network.metrics import (
    compute_structural_metrics,
    compute_local_metrics,
    detect_communities,
)
from vehicle_theft_network.visualization.map_utils import (
    generate_centroid_polygon_dict,
    build_centroid_kdtree,
    plot_communities_on_map,
    plot_metric_map,
    plot_cell_and_neighbours,
)
from vehicle_theft_network.visualization.plots import visualize_two_series_events

data_cfg = DataConfig()
ev_cfg   = EventSyncConfig()

## 2. Load Pre-Computed Data

### 2.1 Monte Carlo simulation counts

Each entry `(i, j)` counts how many of the 1 000 surrogates showed synchronization between cells *i* and *j*. Pairs exceeding 950 (95%) are considered significant.

In [1]:
monte_carlo = load_npy(data_cfg.monte_carlo_path)
print(f'Monte Carlo matrix shape: {monte_carlo.shape}')
monte_carlo

Monte Carlo matrix shape: (1336, 1336)


array([[  0, 895, 382, ..., 328, 347,   0],
       [895,   0, 495, ..., 315,  32, 136],
       [382, 495,   0, ..., 897, 915, 185],
       ...,
       [328, 315, 897, ...,   0, 518, 342],
       [347,  32, 915, ..., 518,   0, 535],
       [  0, 136, 185, ..., 342, 535,   0]], dtype=int16)

### 2.2 Filtered time series (for coordinate labels)

In [ ]:
import pandas as pd
hourly_ts = pd.read_pickle(data_cfg.time_series_path)
print(hourly_ts.shape)
hourly_ts.head()

### 2.3 Grid cells and cell data

In [ ]:
grid_cells    = load_pickle(data_cfg.grid_cells_path)
cell_data_dict = load_pickle(data_cfg.cell_data_dict_path)
print(f'Grid cells: {len(grid_cells):,} rows, {grid_cells["geometry"].nunique():,} unique polygons')
print(f'Cell data dict: {len(cell_data_dict):,} cells')

## 3. Build Adjacency Matrix

Threshold: pairs must exceed `n_surr * significance_level` surrogate scores.

**Thesis parameters:** `n_surr = 1000`, `significance_level = 0.95` → threshold = 950.

In [2]:
adjacency = build_adjacency_matrix(
    monte_carlo,
    n_surr=ev_cfg.n_surr,
    significance_level=ev_cfg.significance_level,
)
print(f'Edges: {adjacency.sum() // 2:,}')
adjacency

Edges: 18,541


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

## 4. Build Complex Network

In [ ]:
coord_labels = get_coord_labels(hourly_ts)
G = build_network(adjacency, coord_labels)
print(f'Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}')

## 5. Structural Metrics

Global / structural properties of the network.

In [3]:
import networkx as nx

structural = compute_structural_metrics(G)

# Diameter and radius (computed separately for the main component)
components = sorted(nx.connected_components(G), key=len, reverse=True)
main = G.subgraph(components[0])
print(f'Link Density: {structural["link_density"]:.5f}')
print(f'Average Shortest Path Length: {structural["avg_shortest_path_length"]:.5f}')
print(f'Radius (main component): {nx.radius(main)}')
print(f'Diameter (main component): {nx.diameter(main)}')
print(f'Transitivity: {structural["transitivity"]:.5f}')
print(f'Assortativity: {structural["assortativity"]:.5f}')
print(f'Global Clustering Coefficient: {structural["global_clustering"]:.5f}')
print(f'Average Degree: {structural["avg_degree"]:.3f}')

Link Density: 0.02084931261073359
Average Shortest Path Length: 0.14234099012538556
Radius (main component): 3
Diameter (main component): 6
Transitivity: 0.08527074194750338
Assortativity: -0.016011524761489062
Global Clustering Coefficient: 0.104592807353886
Average Degree: 27.834


## 6. Local Metrics

Node-level centrality measures.

In [ ]:
local = compute_local_metrics(G)

### 6.1 Hub identification

The node maximising all four centrality measures simultaneously.

In [4]:
measures = {
    'Degree Centrality':      local['degree_centrality'],
    'Closeness Centrality':   local['closeness_centrality'],
    'Betweenness Centrality': local['betweenness_centrality'],
    'Eigenvector Centrality': local['eigenvector_centrality'],
}
print('Central Nodes (all centrality measures agree):')
for name, vals in measures.items():
    hub = max(vals, key=vals.get)
    print(f'  {name:<28}: {coord_labels[hub]}')

Central Nodes (all centrality measures agree):
  Degree Centrality      : (-46.605, -23.561)
  Closeness Centrality   : (-46.605, -23.561)
  Betweenness Centrality : (-46.605, -23.561)
  Eigenvector Centrality : (-46.605, -23.561)


### 6.2 Neighbourhood of the hub

In [ ]:
hub_node = max(local['degree_centrality'], key=local['degree_centrality'].get)
hub_coords = coord_labels[hub_node]
hub_neighbours = [coord_labels[n] for n in G.neighbors(hub_node)]
print(f'Hub: {hub_coords}  |  degree = {local["degree"][hub_node]}')
print(f'First 5 neighbours: {hub_neighbours[:5]}')

## 7. Community Detection

The Girvan–Newman algorithm iteratively removes the edge with the highest betweenness centrality, splitting the graph into communities at the first cut.

In [5]:
communities = detect_communities(G)
sizes = sorted([len(v) for v in communities.values()], reverse=True)
print(f'{len(communities)} communities detected')
for i, (cid, nodes) in enumerate(sorted(communities.items(), key=lambda x: -len(x[1]))):
    if i < 3 or len(nodes) > 1:
        print(f'  Community {cid}: {len(nodes)} node(s)')

19 communities detected
  Community 0: 1317 nodes (large cluster)
  Community 1:    2 nodes
  Communities 2-18: 1 node each (isolated)


## 8. Visualisations

All maps are interactive Folium HTML files. Run the cells to display them inline or use `map.save('output.html')` to export.

> **Note:** The map outputs below were generated during the original thesis run in Google Colab; they are too large to embed in the notebook.

### 8.1 Spatial structures (grid topology)

In [ ]:
polygon_dict = generate_centroid_polygon_dict(grid_cells)
kdtree, centroids = build_centroid_kdtree(polygon_dict)

### 8.2 Community map

In [6]:
communities_map = plot_communities_on_map(
    community_nodes_dict=communities,
    coord_labels=coord_labels,
    polygon_dict=polygon_dict,
    cell_data_dict=cell_data_dict,
    kdtree=kdtree,
    centroids=centroids,
    zoom_level=10,
    seed_value=24,
)
# communities_map.save('outputs/communities.html')
communities_map

Output hidden; open in https://colab.research.google.com to view.

### 8.3 Hub and neighbourhood map

In [7]:
edge_coords = {}
for u, v in G.edges():
    u_c, v_c = coord_labels[u], coord_labels[v]
    edge_coords.setdefault(u_c, []).append(v_c)
    edge_coords.setdefault(v_c, []).append(u_c)

hub_map = plot_cell_and_neighbours(
    key=hub_coords,
    values=edge_coords[hub_coords],
    cell_data_dict=cell_data_dict,
    polygon_dict=polygon_dict,
    kdtree=kdtree,
    centroids=centroids,
    edge_opacity=0.5,
)
# hub_map.save('outputs/hub_neighbours.html')
hub_map

<folium.folium.Map object>

### 8.4 Degree map (polygon choropleth)

In [8]:
degree_map = plot_metric_map(
    coord_labels=coord_labels,
    metric_values=local['degree'],
    polygon_dict=polygon_dict,
    cell_data_dict=cell_data_dict,
    kdtree=kdtree,
    centroids=centroids,
    visualization_type='polygon',
    metric_name='Degree',
    metric_precision=0,
)
# degree_map.save('outputs/degree_polygon.html')
degree_map

Output hidden; open in https://colab.research.google.com to view.

### 8.5 Centrality maps

In [ ]:
for metric_name, metric_values in [
    ('Betweenness Centrality', local['betweenness_centrality']),
    ('Closeness Centrality',   local['closeness_centrality']),
    ('Eigenvector Centrality', local['eigenvector_centrality']),
    ('Clustering Coefficient', local['clustering']),
]:
    m = plot_metric_map(
        coord_labels=coord_labels,
        metric_values=metric_values,
        polygon_dict=polygon_dict,
        cell_data_dict=cell_data_dict,
        kdtree=kdtree,
        centroids=centroids,
        visualization_type='marker',
        metric_name=metric_name,
    )
    # m.save(f'outputs/{metric_name.lower().replace(" ", "_")}.html')

### 8.6 Comparing two event series

In [ ]:
import pandas as pd
hourly_ts = pd.read_pickle(data_cfg.time_series_path)

# Hub cell vs. a distant cell (adjust coordinates as needed)
visualize_two_series_events(
    df=hourly_ts,
    cell_1_coords=(-46.605, -23.561),
    cell_2_coords=(-47.391, -20.539),
    event_color_1='blue',
    event_color_2='red',
    non_event_color='green',
    title='Comparison of Events in Two Cells',
)